<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day07_foundation_revision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7: Foundation Revision
**Course:** BO CDA 205, Machine Learning Techniques
**Dataset:** German Credit (1000 customers, 21 columns)
**Syllabus link:** Foundation (data loading, EDA, encoding, scaling, splitting)

> **Goal of today:** Put the whole preprocessing workflow into reusable functions, with safety checks, so every later notebook can start from the same clean data.

## Foundation Recap (Day 1 to 6)

| Day | Topic | Key takeaway |
|---|---|---|
| 1 | Data loading | Use `shape`, `info`, `head` and `tail`. The dataset has 1000 rows and 21 columns, with no missing values |
| 2 | Understanding the data | Classes are imbalanced (70% good, 30% bad). `credit_amount` is right-skewed (mean 3271 vs median 2319.5) |
| 3 | EDA with graphs | Use histograms, boxplots, `groupby` and `crosstab(normalize='index')`. Compare percentages, not raw counts |
| 4 | Correlation | r ranges from -1 to +1. Check features against each other (redundancy) and against the target (usefulness) |
| 5 | Encoding | One-hot for nominal features, label encoding only for the target, ordinal encoding only when a real order exists (and we specify it) |
| 6 | Split and scaling | Split first, then fit on the training set only. On the test set, use `transform` only |

### Five rules to remember
1. **Split first**, then learn anything (scaler, encoder, feature selection) from the training set only
2. **`fit_transform` on train, `transform` on test** to avoid data leakage
3. **Accuracy alone is not enough.** The "always good" baseline scores 70% accuracy with 0 recall for bad customers
4. **No label encoding on nominal features**, because it creates a fake order
5. **Use `random_state=42` and `stratify=y`** so results are reproducible and class ratios are preserved

## 1. Preprocessing as Functions

A **function** is a named block of code we can call again and again. We write the workflow once and reuse it in every later notebook, so the split and preprocessing are always identical.

| Function | Job |
|---|---|
| `load_data()` | download the data, return features `X` and target `y` |
| `split_data()` | stratified 80/20 train and test split |
| `build_preprocessor()` | scale numeric columns, one-hot encode categorical columns |

In [13]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


def load_data():
    """Load German Credit data. Returns X (features) and y (bad = 1, good = 0)."""
    data = fetch_openml(name="credit-g", version=1, as_frame=True)
    df = data.frame
    X = df.drop(columns=['class'])
    # astype(int) makes y plain 0/1 numbers instead of a category type
    y = df['class'].map({'good': 0, 'bad': 1}).astype(int)
    return X, y


def split_data(X, y, test_size=0.2, seed=42):
    """Stratified train-test split with a fixed seed so results are reproducible."""
    return train_test_split(X, y, test_size=test_size,
                            random_state=seed, stratify=y)


def build_preprocessor(X):
    """Scale numeric columns and one-hot encode categorical columns."""
    num_cols = X.select_dtypes(include='number').columns.tolist()
    cat_cols = X.select_dtypes(exclude='number').columns.tolist()

    pre = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ],
        sparse_threshold=0
    )
    pre.set_output(transform="pandas")   # keep column names in the output
    return pre

## 2. Running the Workflow

The order never changes: **load, split, fit the preprocessor on train only, transform both.**

In [14]:
X, y = load_data()
X_train, X_test, y_train, y_test = split_data(X, y)

preprocess = build_preprocessor(X_train)
X_train_p = preprocess.fit_transform(X_train)   # learn from train and transform
X_test_p = preprocess.transform(X_test)         # apply the same learning to test

print("Train:", X_train_p.shape, "| Test:", X_test_p.shape)

Train: (800, 61) | Test: (200, 61)


## 3. Safety Checks with `assert`

`assert condition` stops the program with an error if the condition is **False**. It is a cheap way to catch silent mistakes before they spoil results.

In [15]:
# 1. Sizes: 800 train rows, 200 test rows
assert X_train_p.shape[0] == 800 and X_test_p.shape[0] == 200

# 2. Train and test must have exactly the same columns in the same order
assert list(X_train_p.columns) == list(X_test_p.columns)

# 3. No missing values anywhere
assert X_train_p.isnull().sum().sum() == 0
assert X_test_p.isnull().sum().sum() == 0

# 4. Everything must be numeric
assert X_train_p.select_dtypes(exclude='number').shape[1] == 0

# 5. Class ratio must be about the same in train and test (stratify worked)
assert abs(y_train.mean() - y_test.mean()) < 0.02

# 6. Scaling check: numeric columns in train must have mean about 0
assert abs(X_train_p['num__age'].mean()) < 1e-6

print("All checks passed")

All checks passed


In [16]:
summary = pd.DataFrame({
    'rows': [len(X_train_p), len(X_test_p)],
    'columns': [X_train_p.shape[1], X_test_p.shape[1]],
    'bad_rate': [y_train.mean().round(3), y_test.mean().round(3)]
}, index=['train', 'test'])
print(summary)

       rows  columns  bad_rate
train   800       61       0.3
test    200       61       0.3


In [17]:
import random
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Each question: the FIRST option in the list is the correct one.
# The options are shuffled before showing, so the position of the answer changes every run.
questions = [
    {
        "q": "70% of customers are good. A model predicts 'good' for everyone. What accuracy does it get, and how many bad customers does it catch?",
        "options": [
            "70% accuracy, catches 0 bad customers",
            "70% accuracy, catches 30% of bad customers",
            "100% accuracy, catches all bad customers",
            "30% accuracy, catches 0 bad customers",
        ],
        "explain": "It is right for all 700 good customers (70%) but never predicts 'bad', so recall for the bad class is 0. Accuracy alone is not enough.",
        "revise": "Day 2 and Day 6",
    },
    {
        "q": "credit_amount has mean 3271 and median 2319.5. What does this tell us?",
        "options": [
            "Right-skewed: a few very large loans pull the average up",
            "Left-skewed",
            "Perfectly symmetric",
            "The column has missing values",
        ],
        "explain": "When the mean is much larger than the median, a few very large values pull the mean upward. This is a right skew.",
        "revise": "Day 2",
    },
    {
        "q": "Why is label encoding (0, 1, 2, ...) wrong for the 'purpose' column?",
        "options": [
            "The model sees a fake order and distance between categories",
            "Label encoding only works on numeric data",
            "It creates too many columns",
            "It creates missing values",
        ],
        "explain": "'purpose' is nominal (no natural order). KNN and SVM would treat 2 as larger than 1 and farther from 0. One-hot encoding avoids this.",
        "revise": "Day 5",
    },
    {
        "q": "How should a scaler be applied to train and test data?",
        "options": [
            "fit_transform on train, only transform on test",
            "fit_transform separately on both",
            "Fit on the full data, then split",
            "fit_transform on test, transform on train",
        ],
        "explain": "The scaler must learn its mean and standard deviation from the training set only. Fitting on test data leaks information (data leakage).",
        "revise": "Day 6",
    },
    {
        "q": "Two features have a correlation of r = 0.98. What do you do?",
        "options": [
            "Consider dropping one, since both carry almost the same information",
            "Always keep both, more features are better",
            "Drop both",
            "Nothing, correlation is only for graphs",
        ],
        "explain": "A correlation this high means redundancy (multicollinearity). Dropping one loses very little information and keeps the model simpler.",
        "revise": "Day 4",
    },
    {
        "q": "Which of these models does NOT need scaling?",
        "options": [
            "Decision Tree",
            "KNN",
            "SVM",
            "K-Means",
        ],
        "explain": "A Decision Tree splits on one feature at a time with a threshold (for example age < 30), so feature ranges do not matter. KNN, SVM and K-Means are distance-based.",
        "revise": "Day 6",
    },
]

# Build the questions on screen
radios = []
for i, item in enumerate(questions):
    shuffled = item["options"][:]          # copy of the list
    random.shuffle(shuffled)               # random order each run
    radio = widgets.RadioButtons(
        options=shuffled,
        value=None,                        # nothing selected at the start
        layout=widgets.Layout(width="max-content"),
    )
    display(HTML(f"<h4 style='margin-bottom:2px'>Q{i+1}. {item['q']}</h4>"))
    display(radio)
    radios.append(radio)

button = widgets.Button(description="Check answers", button_style="primary")
output = widgets.Output()


def check_answers(_):
    with output:
        clear_output()
        score = 0
        unanswered = 0
        for i, (item, radio) in enumerate(zip(questions, radios)):
            correct_answer = item["options"][0]
            if radio.value is None:
                unanswered += 1
                display(HTML(f"<p style='color:#b8860b'>Q{i+1}: not answered</p>"))
            elif radio.value == correct_answer:
                score += 1
                display(HTML(f"<p style='color:#1b7f3b'><b>Q{i+1}: correct.</b> {item['explain']}</p>"))
            else:
                display(HTML(
                    f"<p style='color:#c0392b'><b>Q{i+1}: wrong.</b> "
                    f"Correct answer: <i>{correct_answer}</i><br>{item['explain']}<br>"
                    f"<b>Revise:</b> {item['revise']}</p>"
                ))
        display(HTML(f"<h3>Score: {score} / {len(questions)}</h3>"))
        if unanswered:
            display(HTML(f"<p>{unanswered} question(s) left unanswered.</p>"))


button.on_click(check_answers)
display(button, output)

RadioButtons(layout=Layout(width='max-content'), options=('70% accuracy, catches 30% of bad customers', '30% a…

RadioButtons(layout=Layout(width='max-content'), options=('Left-skewed', 'Right-skewed: a few very large loans…

RadioButtons(layout=Layout(width='max-content'), options=('It creates missing values', 'Label encoding only wo…

RadioButtons(layout=Layout(width='max-content'), options=('fit_transform separately on both', 'fit_transform o…

RadioButtons(layout=Layout(width='max-content'), options=('Drop both', 'Nothing, correlation is only for graph…

RadioButtons(layout=Layout(width='max-content'), options=('K-Means', 'SVM', 'KNN', 'Decision Tree'), value=Non…

Button(button_style='primary', description='Check answers', style=ButtonStyle())

Output()